Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


Load Datasets

In [ ]:
students = pd.read_csv("../Cleaned_Data/Students_Cleaned.csv")
courses = pd.read_csv("../Cleaned_Data/Course_Catalogue_Cleaned.csv")
interactions = pd.read_csv(
    "../Cleaned_Data/Student_Course_Interactions_Cleaned.csv"
)


Understand the datasets

In [ ]:
print("Student Columns:")
print(students.columns.tolist())

print("\nInteraction Columns:")
print(interactions.columns.tolist())

print("\nCourse Columns:")
print(courses.columns.tolist())

interactions.columns = interactions.columns.str.strip()
students.columns = students.columns.str.strip()
courses.columns = courses.columns.str.strip()


Merge datasets

In [ ]:
# Rename course column
courses = courses.rename(
    columns={"Course Code": "Course_Code"}
)

# Merge student information
df = pd.merge(
    interactions[["Student_ID", "Course_Code", "Suitability_Label"]],
    students,
    on="Student_ID",
    how="inner"
)

# Merge course information
df = pd.merge(
    df,
    courses,
    on="Course_Code",
    how="left"
)

print("Final Dataset Shape:", df.shape)
df.head()


Data preparation

In [ ]:
# Remove missing target values
df = df.dropna(subset=["Suitability_Label"])

# Remove duplicate records
df = df.drop_duplicates()

# Remove identifiers and leakage-prone columns
columns_to_drop = [
    "Student_ID",
    "Course_Code",
    "Selected_Course_Code",
    "Selected_Exploratory_Course",
    "Selection_Confidence"
]

df = df.drop(
    columns=[col for col in columns_to_drop if col in df.columns],
    errors="ignore"
)

# Separate features and target
X = df.drop("Suitability_Label", axis=1)
y = df["Suitability_Label"]

print("Features Shape:", X.shape)
print("Target Distribution:")
print(y.value_counts())


Identify numerical and categorical columns

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical Features:", numeric_features)
print("\nCategorical Features:", categorical_features)


Preprocessing

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


Random Forest Model

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest Model Trained Successfully!")


Model prediction

In [ ]:
y_pred = rf_model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

results.head(10)


Model evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Confusion matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=rf_model.classes_
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=rf_model.classes_,
    yticklabels=rf_model.classes_
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Random Forest - Confusion Matrix")
plt.show()


Feature importance

In [ ]:
feature_names = rf_model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = rf_model.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

importance_df.head(15)


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance_df.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Important Features - Random Forest")
plt.show()


Conclusion

In [ ]:
## Conclusion

#The Random Forest Classifier was implemented to predict course
#suitability based on student and course-related characteristics.

#The model was evaluated using accuracy, precision, recall,
#F1-score and a confusion matrix.

#Feature importance was also analyzed to identify the factors
#contributing most to course suitability predictions.
